# DataMate baseline: Gemma 4 E4B
Chọn **Runtime → Change runtime type → GPU** (mục tiêu T4 16 GB). Chạy các ô theo thứ tự. Mặc định chỉ chạy 5 câu train, không fine-tune. Model được nạp bằng Hugging Face Transformers với NF4 4-bit.

Luồng: chọn bảng/cột → sinh Python → thực thi toàn bộ dữ liệu của bảng → sửa lỗi tối đa 3 lần. Đối chiếu đáp án chỉ diễn ra sau dự đoán. Code sinh ra được kiểm tra và chạy trong subprocess có timeout; **đây không phải sandbox bảo mật hoàn chỉnh**. Chỉ dùng trên runtime thử nghiệm và dữ liệu bạn tin cậy.

[Paper](https://aclanthology.org/2025.semeval-1.188/) · [Model card](https://huggingface.co/google/gemma-4-E4B-it)


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

CODE_REF = 'codex/datamate-baseline'  # Có thể thay bằng commit SHA để cố định mã.
DATA_ROOT = Path('/content/drive/MyDrive/datasets/split_colab_seed_42')
SPLIT = 'train'
LIMIT = 5  # None = toàn bộ split; 5 là chạy thử.
SEED = 42
MAX_REPAIRS = 3
EXEC_TIMEOUT = 60
MAX_INPUT_TOKENS = 12288
MAX_NEW_TOKENS = 2048
MODEL_ID = 'google/gemma-4-E4B-it'
MODEL_REVISION = 'main'  # Resume kiểm tra cả revision thực tế đã tải.
ENABLE_THINKING = False
RUN_NAME = 'train_' + datetime.now(ZoneInfo('Asia/Saigon')).strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('/content/drive/MyDrive/datamate_runs') / RUN_NAME
RESUME = False  # Khi tiếp tục: đặt OUTPUT_DIR đúng thư mục cũ và RESUME=True.
print('Dataset:', DATA_ROOT / SPLIT)
print('Output:', OUTPUT_DIR)


In [ ]:
import json
import subprocess
import sys
import urllib.request
from urllib.parse import quote

REPO = 'luwukien/zalo-ai-challenge'
request = urllib.request.Request(
    f'https://api.github.com/repos/{REPO}/commits/{quote(CODE_REF, safe="")}',
    headers={'User-Agent': 'datamate-colab'})
with urllib.request.urlopen(request, timeout=60) as response:
    CODE_SHA = json.load(response)['sha']
LOCAL_CODE = Path('/content/datamate_baseline')
LOCAL_CODE.mkdir(parents=True, exist_ok=True)
for filename in ('datamate_baseline.py', 'requirements-datamate.txt'):
    url = f'https://raw.githubusercontent.com/{REPO}/{CODE_SHA}/scripts/{filename}'
    with urllib.request.urlopen(url, timeout=60) as response:
        (LOCAL_CODE / filename).write_bytes(response.read())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade',
                       '-r', str(LOCAL_CODE / 'requirements-datamate.txt')])
print('Code commit:', CODE_SHA)
print('Nếu Colab yêu cầu restart runtime, hãy restart rồi chạy lại các ô.')


In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
# HF_TOKEN là tuỳ chọn trong Colab Secrets; không ghi token vào notebook/log.
try:
    token = userdata.get('HF_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = None
if token:
    os.environ['HF_TOKEN'] = token
del token
if not (DATA_ROOT / SPLIT / 'questions.jsonl').is_file():
    raise FileNotFoundError(f'Sửa DATA_ROOT: không thấy {DATA_ROOT / SPLIT / "questions.jsonl"}')


In [ ]:
import importlib
sys.path.insert(0, str(LOCAL_CODE))
import datamate_baseline as dm
import pandas as pd
from IPython.display import display, Markdown

config = dm.Config(data_root=str(DATA_ROOT), split=SPLIT, limit=LIMIT, seed=SEED,
                   max_repairs=MAX_REPAIRS, exec_timeout=EXEC_TIMEOUT,
                   max_input_tokens=MAX_INPUT_TOKENS, max_new_tokens=MAX_NEW_TOKENS,
                   model_id=MODEL_ID, revision=MODEL_REVISION, enable_thinking=ENABLE_THINKING,
                   output_dir=str(OUTPUT_DIR), resume=RESUME)
# Một model dùng lại cho train và validation, không nạp hai bản lên GPU.
MODEL = dm.HFModel(config)
display(MODEL.info)


In [ ]:
def show_question(record):
    display(Markdown('### ' + record['id'] + ' — ' + record['question']))
    for attempt in record['attempts']:
        if 'code' in attempt:
            print('Code attempt', attempt['attempt'])
            print(attempt['code'])
        error = attempt.get('error') or attempt.get('execution', {}).get('error')
        if error:
            print('ERROR:', error)
    print('Timing:', record['timing'])

summary = dm.run_baseline(config, model=MODEL, on_result=show_question)
print('Trạng thái:', summary['status'])
print('Đã hoàn thành:', summary['completed'], '/', summary['requested'])
if summary.get('error'):
    print(summary['error'])
print('Kết quả:', summary['output_dir'])


In [ ]:
def show_run(output_dir):
    output_dir = Path(output_dir)
    report = json.loads((output_dir / 'summary.json').read_text(encoding='utf-8'))
    refs = {r['id']: r for r in report.get('reference_evaluation', {}).get('comparisons', [])}
    rows = []
    for r in dm.read_jsonl(output_dir / 'trace.jsonl'):
        reference = refs.get(r['id'], {})
        rows.append({'id': r['id'], 'question': r['question'],
                     'prediction': r.get('prediction'), 'reference': reference.get('reference'),
                     'match_tham_khao': reference.get('match'), 'status': r['status'],
                     'repairs': r.get('repairs_used'), 'seconds': r['timing'].get('total_seconds')})
    display(pd.DataFrame(rows))
    print('Đối chiếu tham khảo, không thay thế bộ chấm chính thức.')

show_run(OUTPUT_DIR)


## Chạy validation khi sẵn sàng
Ô dưới mặc định **không chạy**. Đặt `RUN_VALIDATION=True`, chọn `VALIDATION_LIMIT=20` hoặc `None` cho toàn bộ. Validation có thư mục kết quả riêng và dùng lại model đã nạp.

Để tiếp tục sau mất kết nối: dùng cùng CODE_REF/MODEL_REVISION và cấu hình, đặt OUTPUT_DIR về thư mục cũ, RESUME=True. Có thể tăng LIMIT; không được đổi seed, prompt, model hoặc dữ liệu. Câu đã hoàn thành được giữ nguyên; câu đang dở được chạy lại. Nếu OOM, giảm MAX_INPUT_TOKENS/MAX_NEW_TOKENS và tạo lượt chạy mới vì cấu hình đã thay đổi.


In [ ]:
from dataclasses import replace

RUN_VALIDATION = False
VALIDATION_LIMIT = 20  # None để chạy toàn bộ validation.
if RUN_VALIDATION:
    validation_output = OUTPUT_DIR.parent / ('validation_' + datetime.now(ZoneInfo('Asia/Saigon')).strftime('%Y%m%d_%H%M%S'))
    validation_config = replace(config, split='validation', limit=VALIDATION_LIMIT,
                                output_dir=str(validation_output), resume=False)
    validation_summary = dm.run_baseline(validation_config, model=MODEL, on_result=show_question)
    show_run(validation_output)
else:
    print('Validation chưa chạy. Đổi RUN_VALIDATION=True khi muốn chạy.')
